# Harvard OCT B-Scan Dataset Processor

Downloads all three Harvard Ophthalmology AI Lab OCT B-scan datasets from Hugging Face, extracts only `oct_bscans` + `glaucoma` labels (strips demographics, fundus images, RNFLT maps, clinical metadata), and pushes a unified compressed dataset back to HF Hub.

| Dataset | Samples | B-scan Shape |
|---|---|---|
| Harvard-GF | 3,300 | (200, 200, 200) uint8 |
| FairFedMed-Oph | 15,165 | (200, 200, 200) uint8 |
| FairGenMed | 10,052 | (200, 200, 200) uint8 |
| **Combined** | **~28,500** | (200, 200, 200) uint8 |

**Runtime:** ~2-4 hours on Colab Pro (L4/A100). Uses Colab SSD (~80 GB free).
**Output:** `Tqhuyen/harvard-oct-glaucoma-200` on Hugging Face.

## Cell 1: Mount Drive + Install Dependencies

In [ ]:
# @title Mount Google Drive (for caching checkpoints, optional)

from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets huggingface_hub tqdm scipy pandas

import os, zipfile, io, json, shutil, random, time
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download, create_repo, upload_folder
from google.colab import userdata

# --- CONFIG ---
HF_USER = "Tqhuyen"          # CHANGE to your HF username
OUTPUT_REPO = f"{HF_USER}/harvard-oct-glaucoma-200"

# Get HF token from Colab Secrets (key icon on left → HF_TOKEN → hf_xxx)
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    api = HfApi(token=HF_TOKEN)
    print(f"HF_TOKEN loaded. User: {api.whoami()['name']}")
except Exception as e:
    print(f"ERROR: Could not load HF_TOKEN from Colab Secrets: {e}")
    print("Go to key icon (left sidebar) → Add secret → name: HF_TOKEN → value: hf_xxx")
    raise

random.seed(42)
np.random.seed(42)
print(f"Output repo: {OUTPUT_REPO}")

## Cell 2: Download + Extract Harvard-GF (3,300 samples)

In [ ]:
# @title Process Harvard-GF

DATASET = "harvardairobotics/Harvard-GF"
ZIP_PATH = "Dataset/dataset.zip"
OUT_DIR = Path("/content/harvard_gf_extracted")

print(f"[1/2] Downloading {DATASET}/{ZIP_PATH}...")
t0 = time.time()
zip_path = hf_hub_download(
    repo_id=DATASET, filename=ZIP_PATH, repo_type="dataset"
)
print(f"      Downloaded in {(time.time() - t0) / 60:.1f} min → {zip_path}")

# Load metadata CSV for labels and splits
csv_path = hf_hub_download(
    repo_id=DATASET, filename="ReadMe/data_summary.csv", repo_type="dataset"
)
meta = pd.read_csv(csv_path)
label_map = dict(zip(meta["filename"], meta["glaucoma"].map({"yes": 1, "no": 0})))
split_map = dict(zip(meta["filename"], meta["use"]))
print(f"      Metadata: {len(meta)} rows, {sum(v == 1 for v in label_map.values())} glaucoma-positive")

os.makedirs(OUT_DIR, exist_ok=True)
count, errors = 0, 0

print(f"[2/2] Extracting B-scans + labels...")
with zipfile.ZipFile(zip_path, "r") as zf:
    npz_files = [n for n in zf.namelist() if n.endswith(".npz")]
    for fname in tqdm(npz_files, desc="Harvard-GF"):
        try:
            data = zf.read(fname)
            with io.BytesIO(data) as bio:
                npz = np.load(bio, allow_pickle=True)
                oct_scan = np.asarray(npz["oct_bscans"], dtype=np.uint8)
                label = label_map.get(fname, -1)
            np.savez_compressed(str(OUT_DIR / fname), oct_bscans=oct_scan, glaucoma=label)
            count += 1
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"      WARN: {fname}: {e}")

sz = sum(f.stat().st_size for f in OUT_DIR.glob("*.npz"))
print(f"Done: {count} files, {sz / 1e9:.1f} GB, {errors} errors")

# Free space
os.remove(zip_path)
print("Zip removed from SSD.")

## Cell 3: Download + Extract FairFedMed-Oph (15,165 samples)

In [ ]:
# @title Process FairFedMed-Oph

DATASET = "harvardairobotics/FairFedMed"
ZIP_PATH = "FairFedMed-Oph/Dataset/dataset.zip"
OUT_DIR = Path("/content/fairfedmed_extracted")

print(f"[1/2] Downloading {DATASET}/{ZIP_PATH}...")
t0 = time.time()
zip_path = hf_hub_download(
    repo_id=DATASET, filename=ZIP_PATH, repo_type="dataset"
)
print(f"      Downloaded in {(time.time() - t0) / 60:.1f} min")

csv_path = hf_hub_download(
    repo_id=DATASET, filename="FairFedMed-Oph/ReadMe/data_summary.csv", repo_type="dataset"
)
meta = pd.read_csv(csv_path)
label_map = dict(zip(meta["filename"], meta["glaucoma"].map({"yes": 1, "no": 0})))
print(f"      Metadata: {len(meta)} rows")

os.makedirs(OUT_DIR, exist_ok=True)
count, errors = 0, 0

print(f"[2/2] Extracting B-scans + labels...")
with zipfile.ZipFile(zip_path, "r") as zf:
    npz_files = [n for n in zf.namelist() if n.endswith(".npz")]
    for fname in tqdm(npz_files, desc="FairFedMed-Oph"):
        try:
            data = zf.read(fname)
            with io.BytesIO(data) as bio:
                npz = np.load(bio, allow_pickle=True)
                oct_scan = np.asarray(npz["oct_bscans"], dtype=np.uint8)
                label = label_map.get(fname, -1)
            np.savez_compressed(str(OUT_DIR / fname), oct_bscans=oct_scan, glaucoma=label)
            count += 1
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"      WARN: {fname}: {e}")

sz = sum(f.stat().st_size for f in OUT_DIR.glob("*.npz"))
print(f"Done: {count} files, {sz / 1e9:.1f} GB, {errors} errors")

os.remove(zip_path)
print("Zip removed from SSD.")

## Cell 4: Download + Extract FairGenMed (10,052 samples)

In [ ]:
# @title Process FairGenMed

DATASET = "harvardairobotics/FairGenMed"
SPLITS = {"Training": "training", "Validation": "validation", "Test": "test"}
OUT_DIR = Path("/content/fairgenmed_extracted")

# Load metadata
csv_path = hf_hub_download(
    repo_id=DATASET, filename="ReadMe/data_summary.csv", repo_type="dataset"
)
meta = pd.read_csv(csv_path)
label_map = dict(zip(meta["filename"], meta["glaucoma"].map({"yes": 1, "no": 0})))
print(f"Metadata: {len(meta)} rows")

os.makedirs(OUT_DIR, exist_ok=True)
total, total_size, total_errors = 0, 0, 0

for split_dir, split_name in SPLITS.items():
    zip_filename = f"Dataset/{split_dir}/NPZ.zip"
    print(f"\n[{split_name}] Downloading {DATASET}/{zip_filename}...")
    t0 = time.time()
    zip_path = hf_hub_download(
        repo_id=DATASET, filename=zip_filename, repo_type="dataset"
    )
    print(f"          Downloaded in {(time.time() - t0) / 60:.1f} min")

    split_out = OUT_DIR / split_name
    os.makedirs(split_out, exist_ok=True)
    count, errors = 0, 0

    with zipfile.ZipFile(zip_path, "r") as zf:
        npz_files = [n for n in zf.namelist() if n.endswith(".npz")]
        for fname in tqdm(npz_files, desc=f"FairGenMed/{split_name}"):
            try:
                data = zf.read(fname)
                with io.BytesIO(data) as bio:
                    npz = np.load(bio, allow_pickle=True)
                    oct_scan = np.asarray(npz["oct_bscans"], dtype=np.uint8)
                    label = label_map.get(fname, -1)
                np.savez_compressed(str(split_out / fname), oct_bscans=oct_scan, glaucoma=label)
                count += 1
            except Exception as e:
                errors += 1
                if errors <= 3:
                    print(f"      WARN: {fname}: {e}")

    sz = sum(f.stat().st_size for f in split_out.glob("*.npz"))
    print(f"  {split_name}: {count} files, {sz / 1e9:.1f} GB, {errors} errors")
    total += count
    total_size += sz
    total_errors += errors
    os.remove(zip_path)

print(f"\nTotal FairGenMed: {total} files, {total_size / 1e9:.1f} GB, {total_errors} errors")

## Cell 5: Combine All into Unified Splits

In [ ]:
# @title Combine into unified train/val/test

import random
random.seed(42)

# Destination: per-split volumes
all_scans = {"train": [], "val": [], "test": []}
all_labels = {"train": [], "val": [], "test": []}


def add_from_dir(source_dir, split_col=None):
    """Load extracted .npz files and add to all_scans by split."""
    base = Path(source_dir)
    npz_files = list(base.rglob("*.npz"))
    print(f"  Loading {len(npz_files)} files from {source_dir}...")

    for npz_path in tqdm(npz_files, desc=source_dir):
        try:
            data = dict(np.load(npz_path, allow_pickle=True))
            oct_scan = data["oct_bscans"]
            label = int(data["glaucoma"])

            if split_col:
                split = split_col.get(npz_path.name, "train")
                # Map to our split names
                split = {"training": "train", "validation": "val", "test": "test"}.get(split, "train")
            else:
                split = "pool"  # Will be shuffled later

            if split == "pool":
                continue  # handled below

            all_scans[split].append(oct_scan)
            all_labels[split].append(label)
        except Exception as e:
            pass


# ---- Harvard-GF (has splits: training/validation/test) ----
meta_gf = pd.read_csv(
    hf_hub_download(
        "harvardairobotics/Harvard-GF",
        "ReadMe/data_summary.csv",
        repo_type="dataset"
    )
)
gf_split = dict(zip(meta_gf["filename"], meta_gf["use"]))
print("=== Harvard-GF ===")
add_from_dir("/content/harvard_gf_extracted", gf_split)

# ---- FairGenMed (has splits: Training/Validation/Test) ----
meta_fgm = pd.read_csv(
    hf_hub_download(
        "harvardairobotics/FairGenMed",
        "ReadMe/data_summary.csv",
        repo_type="dataset"
    )
)
fgm_split = dict(zip(meta_fgm["filename"], meta_fgm["use"]))
print("=== FairGenMed ===")
add_from_dir("/content/fairgenmed_extracted", fgm_split)

# ---- FairFedMed (NO splits → shuffle and split 70/10/20) ----
print("=== FairFedMed-Oph (splitting 70/10/20) ===")
ffm_scans, ffm_labels = [], []
for npz_path in tqdm(
    list(Path("/content/fairfedmed_extracted").glob("*.npz")),
    desc="FairFedMed"
):
    try:
        data = dict(np.load(npz_path, allow_pickle=True))
        ffm_scans.append(data["oct_bscans"])
        ffm_labels.append(int(data["glaucoma"]))
    except Exception:
        pass

combined = list(zip(ffm_scans, ffm_labels))
random.shuffle(combined)
n = len(combined)
n_train = int(n * 0.7)
n_val = int(n * 0.1)

for scan, label in combined[:n_train]:
    all_scans["train"].append(scan)
    all_labels["train"].append(label)
for scan, label in combined[n_train : n_train + n_val]:
    all_scans["val"].append(scan)
    all_labels["val"].append(label)
for scan, label in combined[n_train + n_val :]:
    all_scans["test"].append(scan)
    all_labels["test"].append(label)

# ---- Summary ----
print("\n" + "=" * 50)
print(f"{'Split':<10} {'Samples':<12} {'Glaucoma+':<12} {'%'}")
print("-" * 50)
grand = 0
for split in ["train", "val", "test"]:
    n_total = len(all_scans[split])
    n_gl = sum(1 for l in all_labels[split] if l == 1)
    grand += n_total
    print(f"{split:<10} {n_total:<12} {n_gl:<12} {n_gl / n_total * 100:.1f}%")
print("-" * 50)
print(f"{'TOTAL':<10} {grand}")

## Cell 6: Save Unified Dataset + Push to HF Hub

In [ ]:
# @title Push to Hugging Face Hub

UPLOAD_DIR = Path("/content/unified_oct_dataset")
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ---- Save as compressed .npz per split ----
for split_name in ["train", "val", "test"]:
    scans = all_scans[split_name]
    labels = all_labels[split_name]
    if not scans:
        continue

    X = np.stack(scans, axis=0)
    y = np.array(labels, dtype=np.int16)

    out_path = UPLOAD_DIR / f"{split_name}_volumes.npz"
    print(f"Saving {split_name}: X={X.shape}, y={y.shape}...")
    np.savez_compressed(out_path, oct_bscans=X, glaucoma=y)

    sz_gb = out_path.stat().st_size / 1e9
    print(f"  → {out_path.name}: {sz_gb:.2f} GB")

# ---- Save manifest ----
manifest = {
    "description": "Unified Harvard OCT B-scan dataset (200x200x200) for glaucoma classification",
    "total_samples": int(sum(len(all_scans[s]) for s in ["train", "val", "test"])),
    "resolution": [200, 200, 200],
    "dtype": "uint8",
    "channels": 1,
    "classes": {"0": "no_glaucoma", "1": "glaucoma"},
    "source_datasets": [
        "harvardairobotics/Harvard-GF",
        "harvardairobotics/FairFedMed (Oph subset)",
        "harvardairobotics/FairGenMed"
    ],
    "license": "cc-by-nc-nd-4.0",
    "splits": {
        s: {
            "samples": len(all_scans[s]),
            "glaucoma_positive": int(sum(1 for l in all_labels[s] if l == 1)),
            "file": f"{s}_volumes.npz"
        }
        for s in ["train", "val", "test"] if all_scans[s]
    }
}
with open(UPLOAD_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\nManifest saved.")
print(f"Total upload size: {sum(f.stat().st_size for f in UPLOAD_DIR.glob('*.npz')) / 1e9:.1f} GB")

# ---- Upload to HF Hub ----
print(f"\n=== Uploading to https://huggingface.co/datasets/{OUTPUT_REPO} ===")

create_repo(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    private=False,
    exist_ok=True
)

upload_folder(
    repo_id=OUTPUT_REPO,
    folder_path=str(UPLOAD_DIR),
    repo_type="dataset",
    commit_message="Upload unified Harvard OCT B-scans (200x200x200)"
)

print(f"\n✓ DONE! Dataset live at: https://huggingface.co/datasets/{OUTPUT_REPO}")
print(f"  Load it with: load_dataset('{OUTPUT_REPO}')")

## Cell 7 (Optional): Verify + Usage Example

In [ ]:
# @title Verify the uploaded dataset

from datasets import load_dataset

print(f"Loading dataset from HF Hub...")
ds = load_dataset(OUTPUT_REPO)
print(f"Splits: {list(ds.keys())}")

for split in ["train", "val", "test"]:
    if split in ds:
        print(f"\n{split}: {len(ds[split])} samples")
        sample = ds[split][0]
        oct_scan = sample["oct_bscans"]
        label = sample["glaucoma"]
        print(f"  B-scan shape: {oct_scan.shape}")
        print(f"  B-scan dtype: {oct_scan.dtype}")
        print(f"  Label: {label} ({'glaucoma' if label == 1 else 'no glaucoma'})")